# Gray-Box AIA72 — Locked Supplementary-2026 Statistical Analysis

Inferential analysis of the already-frozen one-shot 2026 evaluation. **No tuning, policy reselection, or redesign occurs here.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Supplementary 2026 locked statistical analysis
# ============================================================
#
# IMPORTANT:
# - This notebook does NOT redesign, tune, or select any policy.
# - It analyzes the already-frozen one-shot supplementary_2026 evaluation.
# - Primary comparison remains fallback_v3_q90 vs always-fusion and always-SHARP.
# - Bootstrap CIs and McNemar tests are inferential summaries only.

ROOT = Path("/home/abmoses2000/graybox_aia72_supplementary2026_v3_eval_20261005")
CASES = ROOT / "supplementary2026_policy_cases.csv.gz"
RESULTS = ROOT / "supplementary2026_policy_results.csv"
SUMMARY = ROOT / "summary.json"

ALARM = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv")
OUT = Path("/home/abmoses2000/graybox_aia72_supplementary2026_v3_stats_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

N_BOOT = 5000
SEED = 20261005

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=((y==1)&(pred==1)).sum()
    fp=((y==0)&(pred==1)).sum()
    fn=((y==1)&(pred==0)).sum()
    tn=((y==0)&(pred==0)).sum()
    rec=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    tss=rec+spec-1 if np.isfinite(rec) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    f1=2*prec*rec/(prec+rec) if np.isfinite(prec) and np.isfinite(rec) and (prec+rec)>0 else np.nan
    return {"tss":float(tss),"hss":float(hss),"recall":float(rec),"specificity":float(spec),
            "precision":float(prec),"f1":float(f1)}

def exact_mcnemar_p(b,c):
    # two-sided exact binomial test without scipy dependency
    n=b+c
    if n==0:
        return 1.0
    from math import comb
    k=min(b,c)
    tail=sum(comb(n,i) for i in range(k+1))/(2**n)
    return float(min(1.0,2*tail))

for p in [CASES,RESULTS,SUMMARY,ALARM]:
    if not p.exists():
        raise FileNotFoundError(p)

summary=json.loads(SUMMARY.read_text())
assert summary["status"]=="SUPPLEMENTARY_2026_FALLBACK_V3_CLEAN_EVALUATION_COMPLETE"
assert summary["policy_reselection_performed"] is False
assert summary["threshold_tuning_performed"] is False
assert summary["this_role_was_reserved_as_clean_future_evaluation"] is True

df=pd.read_csv(CASES,low_memory=False)
alarm=pd.read_csv(ALARM)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}

y=df["label"].astype(int).to_numpy()
p_fusion=df["fusion_probability"].astype(float).to_numpy()
p_sharp=df["sharp_probability"].astype(float).to_numpy()
pred_fusion=(p_fusion>=alarm_map["sharp_aia"]).astype(int)
pred_sharp=(p_sharp>=alarm_map["sharp"]).astype(int)
pred_q90=df["fallback_v3_q90_prediction"].astype(int).to_numpy()

assert (pred_q90>=0).all(), "Primary v3 q90 had abstentions/unissued cases in 2026."

systems={
    "fallback_v3_q90":pred_q90,
    "baseline_always_fusion":pred_fusion,
    "baseline_always_sharp":pred_sharp,
}

point=[]
for name,pred in systems.items():
    m=metrics(y,pred)
    point.append({"system":name,**m})
point=pd.DataFrame(point)

# Stratified bootstrap by class to preserve rare-event prevalence structure.
rng=np.random.default_rng(SEED)
idx0=np.flatnonzero(y==0)
idx1=np.flatnonzero(y==1)

boot_rows=[]
for i in range(N_BOOT):
    sample=np.concatenate([
        rng.choice(idx0,size=len(idx0),replace=True),
        rng.choice(idx1,size=len(idx1),replace=True),
    ])
    yy=y[sample]
    vals={}
    for name,pred in systems.items():
        vals[name]=metrics(yy,pred[sample])

    for metric in ["tss","hss","recall","specificity","precision","f1"]:
        boot_rows.append({
            "replicate":i,
            "metric":metric,
            "q90_minus_fusion":vals["fallback_v3_q90"][metric]-vals["baseline_always_fusion"][metric],
            "q90_minus_sharp":vals["fallback_v3_q90"][metric]-vals["baseline_always_sharp"][metric],
        })

boot=pd.DataFrame(boot_rows)

ci_rows=[]
for metric,g in boot.groupby("metric"):
    for comparison,col in [
        ("q90_minus_fusion","q90_minus_fusion"),
        ("q90_minus_sharp","q90_minus_sharp")
    ]:
        arr=g[col].to_numpy(float)
        ci_rows.append({
            "metric":metric,
            "comparison":comparison,
            "mean_bootstrap_delta":float(np.mean(arr)),
            "ci2.5":float(np.quantile(arr,0.025)),
            "ci97.5":float(np.quantile(arr,0.975)),
            "probability_delta_gt_0":float(np.mean(arr>0)),
        })
ci=pd.DataFrame(ci_rows)

# McNemar paired correctness tests.
mcnemar=[]
q_correct=(pred_q90==y)
for name,pred in [
    ("baseline_always_fusion",pred_fusion),
    ("baseline_always_sharp",pred_sharp),
]:
    other=(pred==y)
    b=int((q_correct & ~other).sum())   # q90 correct, comparator wrong
    c=int((~q_correct & other).sum())   # q90 wrong, comparator correct
    mcnemar.append({
        "comparison":f"fallback_v3_q90_vs_{name}",
        "q90_correct_other_wrong":b,
        "q90_wrong_other_correct":c,
        "discordant_total":b+c,
        "exact_two_sided_p":exact_mcnemar_p(b,c),
    })
mcnemar=pd.DataFrame(mcnemar)

point_path=OUT/"point_metrics.csv"
ci_path=OUT/"bootstrap_paired_deltas.csv"
mcnemar_path=OUT/"mcnemar_exact.csv"
protocol_path=OUT/"protocol.json"

point.to_csv(point_path,index=False)
ci.to_csv(ci_path,index=False)
mcnemar.to_csv(mcnemar_path,index=False)

protocol={
    "status":"SUPPLEMENTARY_2026_LOCKED_STATISTICAL_ANALYSIS_COMPLETE",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "policy_reselection_performed":False,
    "threshold_tuning_performed":False,
    "posthoc_policy_redesign_performed":False,
    "primary_policy_id":"fallback_v3_q90",
    "bootstrap":{
        "replicates":N_BOOT,
        "seed":SEED,
        "method":"class-stratified nonparametric bootstrap",
        "interval":"percentile 95%"
    },
    "paired_test":"exact McNemar on per-case correctness",
    "source_sha256":{
        "supplementary2026_policy_cases":sha256(CASES),
        "supplementary2026_policy_results":sha256(RESULTS),
        "supplementary2026_summary":sha256(SUMMARY),
        "alarm_thresholds":sha256(ALARM),
    },
    "output_sha256":{
        point_path.name:sha256(point_path),
        ci_path.name:sha256(ci_path),
        mcnemar_path.name:sha256(mcnemar_path),
    }
}
protocol_path.write_text(json.dumps(protocol,indent=2)+"\n")

print("===== SUPPLEMENTARY 2026 LOCKED STATISTICAL ANALYSIS COMPLETE =====")
print("No tuning/reselection/redesign performed.")
print("\nPoint metrics:")
print(point.to_string(index=False))
print("\nPaired bootstrap deltas (95% CI):")
print(ci.to_string(index=False))
print("\nExact McNemar tests:")
print(mcnemar.to_string(index=False))
print("\nSaved:",protocol_path)
print("Saved:",point_path)
print("Saved:",ci_path)
print("Saved:",mcnemar_path)


===== SUPPLEMENTARY 2026 LOCKED STATISTICAL ANALYSIS COMPLETE =====
No tuning/reselection/redesign performed.

Point metrics:
                system      tss      hss   recall  specificity  precision       f1
       fallback_v3_q90 0.296388 0.317926 0.345766     0.950622   0.406880 0.373842
baseline_always_fusion 0.292216 0.317319 0.339718     0.952499   0.411980 0.372376
 baseline_always_sharp 0.322874 0.267329 0.424395     0.898479   0.290545 0.344941

Paired bootstrap deltas (95% CI):
     metric       comparison  mean_bootstrap_delta     ci2.5    ci97.5  probability_delta_gt_0
         f1 q90_minus_fusion              0.001489 -0.003154  0.006657                  0.7138
         f1  q90_minus_sharp              0.028856  0.013422  0.044046                  1.0000
        hss q90_minus_fusion              0.000625 -0.004641  0.006302                  0.5784
        hss  q90_minus_sharp              0.050575  0.033843  0.066882                  1.0000
  precision q90_minus_fusion    